# Milestone 4 lab, notebook 4 of 8: A gate, and where it stands

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| **4. A gate, and where it stands** (you are here) | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    MILESTONE_DIR, after_the_gates, before_the_clean_gate, gate_results, money, prepare_week, sql,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: A check that is in the way

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

**Aiko Tanaka** manages store `S06`. On Saturday 29 August three customers bought pumpkin soup. The soup was not in the price list yet, so the cleaning step dropped the three lines. $17.97 of her sales disappeared, and no row was left to look at.

Whether a check can see that depends on **where it stands**. Look before the cleaning step, and the lines are still there, and nothing is wrong yet. Look after it, and they are gone.

Until now your checks stood beside the pipeline. From now on they stand **in** it.

A **gate** is a step in the pipeline that runs checks and decides whether the data may go on.

This pipeline has three. Here is the order of its steps, from the DAG file.

In [2]:
dag_file = (MILESTONE_DIR / "dags" / "lotus_nightly_dag.py").read_text()
start = dag_file.rindex("stores_send_files >> load")
print(" ".join(dag_file[start : dag_file.index(")", start)].split()))

stores_send_files >> load >> gate_landing >> clean >> gate_clean >> build >> gate_publish >> publish


| Gate | It stands | It reads |
|---|---|---|
| `gate_landing` | When the files have landed | The raw tables, and the files themselves |
| `gate_clean` | After cleaning | The clean tables |
| `gate_publish` | Before the outputs go out | The outputs of tonight, not yet shown to anyone |

To see what the place does, take **one rule** and ask it in all three places: *the receipts add up to the till's own total.*

## Step 2: At the landing gate, on the raw rows

The raw tables hold every file as it arrived. Add up each store's file, and hold it against the tills.

In [3]:
at_landing = sql("""
    SELECT t.store_id, t.file_day, t.till_cents, coalesce(l.lines_cents, 0) AS lines_cents,
           coalesce(l.lines_cents, 0) - t.till_cents AS difference_cents
    FROM (
        SELECT store_id, CAST(business_date AS DATE) AS file_day,
               sum(CAST(net_total_cents AS BIGINT)) AS till_cents
        FROM raw.till_totals GROUP BY store_id, CAST(business_date AS DATE)
    ) t
    LEFT JOIN (
        SELECT store_id, CAST(substr(_file_name, 17, 10) AS DATE) AS file_day,
               sum(CAST(amount_cents AS BIGINT)) AS lines_cents
        FROM raw.sales_lines GROUP BY store_id, CAST(substr(_file_name, 17, 10) AS DATE)
    ) l ON l.store_id = t.store_id AND l.file_day = t.file_day
    WHERE coalesce(l.lines_cents, 0) <> t.till_cents
    ORDER BY t.file_day, t.store_id
""")
at_landing

,store_id,file_day,till_cents,lines_cents,difference_cents
0,S03,2026-08-24,77720,79618,1898
1,S01,2026-08-25,81977,83784,1807
2,S04,2026-08-25,58742,0,-58742
3,S05,2026-08-27,58084,210152,152068
4,S02,2026-08-28,103428,206856,103428


Five store days. The refund at `S03`, the receipt sent twice at `S01`, the missing file of `S04`, the bananas at `S05`, the file sent twice at `S02`. All of them are faults of the **file**, and here they are caught before anything else has touched the rows.

Aiko's `S06` is not in the list. On the raw rows her day is right: the soup lines are there.

## Step 3: At the clean gate, after cleaning

Now the same rule on the clean tables, as the clean gate finds them.

In [4]:
before_the_clean_gate()
after_cleaning = sql("""
    SELECT t.store_id, t.business_date AS sales_date, sum(t.net_total_cents) AS till_cents,
           coalesce(sum(l.lines_cents), 0) AS lines_cents,
           coalesce(sum(l.lines_cents), 0) - sum(t.net_total_cents) AS difference_cents
    FROM clean.till_totals t
    LEFT JOIN (
        SELECT store_id, till_id, sales_date, sum(amount_cents) AS lines_cents
        FROM clean.sales_lines GROUP BY store_id, till_id, sales_date
    ) l ON l.store_id = t.store_id AND l.till_id = t.till_id AND l.sales_date = t.business_date
    GROUP BY t.store_id, t.business_date
    HAVING coalesce(sum(l.lines_cents), 0) <> sum(t.net_total_cents)
    ORDER BY t.business_date, t.store_id
""")
after_cleaning

The clean tables are rebuilt from the raw rows. No gate has looked at them yet.


,store_id,sales_date,till_cents,lines_cents,difference_cents
0,S03,2026-08-24,77720,79618,1898
1,S01,2026-08-25,81977,83784,1807
2,S04,2026-08-25,58742,0,-58742
3,S05,2026-08-27,58084,210152,152068
4,S02,2026-08-28,103428,206856,103428
5,S06,2026-08-29,99623,97826,-1797
6,S02,2026-08-30,90530,88135,-2395


In [5]:
seen_only_here = after_cleaning.merge(
    at_landing.rename(columns={"file_day": "sales_date"})[["store_id", "sales_date"]].astype(str),
    on=["store_id", "sales_date"], how="left", indicator=True,
)
seen_only_here = seen_only_here[seen_only_here["_merge"] == "left_only"]
for _, row in seen_only_here.iterrows():
    print(f"New after cleaning: {row['store_id']} on {row['sales_date']}, {money(row['difference_cents'])}")

New after cleaning: S06 on 2026-08-29, -$17.97
New after cleaning: S02 on 2026-08-30, -$23.95


Seven store days. The five from before, and two that only exist **after** cleaning.

- `S06` on 29 August, $17.97 short. The cleaning step dropped the soup. This is a fault the pipeline made itself.
- `S02` on 30 August, $23.95 short. The cleaning step takes the day from the till's clock, and one receipt was stamped 03:10 the next morning. By file, the day was complete. By day, it is not.

So a check after cleaning also checks the cleaning.

## Step 4: At the publish gate, on what is left

The clean gate did something about what it found: it set the bad rows aside. (Notebook 6 is about that.) The publish gate then asks the same rule a third time, about the rows that are **left**.

In [6]:
after_the_gates()
sql("""
    SELECT store_id, sales_date, lines_cents, till_cents, rows_set_aside, problem, status
    FROM output.store_day_status
    WHERE status <> 'ok'
    ORDER BY sales_date, store_id
""")

Sunday's night ran again. The warehouse is as the nightly run left it.


,store_id,sales_date,lines_cents,till_cents,rows_set_aside,problem,status
0,S03,2026-08-24,78669,77720,1,rows_set_aside,do not pay yet
1,S04,2026-08-25,0,58742,0,no_sales_file,do not pay yet
2,S05,2026-08-27,57932,58084,1,rows_set_aside,do not pay yet
3,S06,2026-08-29,97826,99623,3,rows_set_aside,do not pay yet
4,S02,2026-08-30,88135,90530,4,rows_set_aside,do not pay yet


Five store days, and the list changed again.

- `S01` on the 25th and `S02` on the 28th are **gone**. The extra copies were set aside, and those two days add up to the tills again.
- The other five do not add up, and each row says why: rows were set aside, or no file came.

Here the rule answers a different question. Not "is a file bad?", and not "did cleaning lose something?". It answers the question of the head of retail operations: **can I pay from this day?**

| The same rule, standing | It found | What it protects |
|---|---|---|
| At landing, on raw rows | 5 store days | Against a bad file, before anything is built on it |
| After cleaning | 7 store days | Also against the pipeline's own mistakes |
| Before publish, on what is left | 5 store days | The reader: nothing goes out that cannot be explained |

Where a gate stands is called its **gate placement**. The same check protects different things in each place.

## Step 5: A rule that only works at the door

Some rules can only be asked in one place. On Tuesday 25 August supplier `SUP-B` renamed its cost column. After the load, the raw table cannot say so: the column is just empty.

Only the file's own first line knows. The load step keeps it, and the landing gate reads it.

In [7]:
sql("""
    SELECT file_name, column_name, position_in_file
    FROM raw.file_columns
    WHERE file_name = 'supplier_prices_SUP-B_2026-08-25.csv'
    ORDER BY position_in_file
""")

,file_name,column_name,position_in_file
0,supplier_prices_SUP-B_2026-08-25.csv,supplier_id,1
1,supplier_prices_SUP-B_2026-08-25.csv,product_id,2
2,supplier_prices_SUP-B_2026-08-25.csv,cost_per_unit,3
3,supplier_prices_SUP-B_2026-08-25.csv,unit,4
4,supplier_prices_SUP-B_2026-08-25.csv,price_date,5


In [8]:
gate_results()[lambda found: found["gate"] == "landing"]

,gate,test_name,action,lens,status,bad_rows,title
0,landing,file_is_missing_a_column,refuse_file,data,fail,1,A file has every column we expect
1,landing,file_has_a_new_column,warn,data,fail,1,A file has no column we did not expect
2,landing,supplier_contract,refuse_file,business,fail,2,A supplier's file keeps the contract


`cost_per_unit` where `unit_cost` was agreed. The landing gate saw it on the night the file arrived, and refused the file. No later gate could have said **why** the cost was empty.

The rule of thumb: put a check as early as it can be asked. A fault caught at the door has touched nothing.

## What you now know

- A **gate** is a step in the pipeline that runs checks and decides whether the data may go on.
- **Gate placement** is where a gate stands: when the files land, after cleaning, or before publish.
- The same rule found 5, then 7, then 5 store days. Each place sees different faults and protects a different thing.
- A check after cleaning also checks the cleaning.
- Some rules can only be asked at the door: the file's own columns.

**Say it in an interview:** "We ran the till reconciliation at three gates. On the raw files it caught five bad store days early. After cleaning it caught two more that our own join and date logic had caused. Before publish it ran on what was left after quarantine, and answered whether each store's day could be paid from."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "Where checks run".

Next: [notebook 5](05_warn_block_or_set_aside.ipynb). A gate found something. Now it must do something.